In [1]:

from tqdm import tqdm
from collections import defaultdict
from cs336_basics.pretokenize import pre_tokenize_parallel

In [5]:
!pip install line_profiler

In [6]:
%load_ext line_profiler

In [7]:
def train_bpe(input_path, vocab_size, special_tokens):
    merges = []
    vocab = [bytes([i]) for i in range(256)]
    vocab += [s.encode("utf-8") for s in special_tokens]

    print("Pretokening .......")
    token_counter = pre_tokenize_parallel(input_path, special_tokens)

    print("Initializing pairs .......")
    merges = []
    vocab = [bytes([i]) for i in range(256)]
    vocab += [s.encode("utf-8") for s in special_tokens]
    init_vocab_len = len(vocab)

    token_pair_set = dict()
    pair_counter = defaultdict(int)
    for token, count in token_counter.items():
        s = set()
        for i in range(len(token) - 1):
            pair = token[i : i + 2]
            pair_counter[pair] += count
            s.add(pair)
        token_pair_set[token] = s

    print("Training BPE .......")
    for epoch in tqdm(range(init_vocab_len + 1, vocab_size)):
        max_pair = max(pair_counter, key=lambda p: (pair_counter[p], p))
        mp0, mp1 = max_pair
        merged_pair = mp0 + mp1
        merges.append(max_pair)

        for token, count in list(token_counter.items()):
            if max_pair not in token_pair_set[token]:
                continue   # no occurrence — token stays unchanged in dict
            modify = False
            new_token = []
            i = 0
            n = len(token)
            while i < n:
                if i + 1 < n and token[i] == mp0 and token[i + 1] == mp1:
                    modify = True
                    if new_token:                       # left-neighbor update
                        left = new_token[-1]
                        lp = (left, mp0)
                        c = pair_counter[lp] - count
                        if c:
                            pair_counter[lp] = c
                        else:
                            del pair_counter[lp]
                        pair_counter[(left, merged_pair)] += count
                    if i + 2 < n:                       # right-neighbor update
                        right = token[i + 2]
                        rp = (mp1, right)
                        c = pair_counter[rp] - count
                        if c:
                            pair_counter[rp] = c
                        else:
                            del pair_counter[rp]
                        pair_counter[(merged_pair, right)] += count
                    new_token.append(merged_pair)
                    i += 2
                else:
                    new_token.append(token[i])
                    i += 1

            # ── in-place update (no new_counter) ──────────────────────────
            if modify:
                new_key = tuple(new_token)
                del token_counter[token]
                del token_pair_set[token]
                token_counter[new_key] = token_counter.get(new_key, 0) + count
                token_pair_set[new_key] = {
                    (new_token[j], new_token[j + 1]) for j in range(len(new_token) - 1)
                }

        del pair_counter[max_pair]   # max_pair gone from all tokens; remove stale count
        vocab.append(merged_pair)
        
    vocab = {i: x for i, x in enumerate(vocab)}
    return vocab, merges

In [8]:
special_tokens = ["<|endoftext|>"]
input_path = "../data/owt_valid.txt"
vocab_size = 300

In [9]:
%lprun -f train_bpe train_bpe(input_path, vocab_size, special_tokens)

Pretokening .......


100%|██████████| 8/8 [00:23<00:00,  3.00s/it]


Initializing pairs .......
Training BPE .......


100%|██████████| 42/42 [00:45<00:00,  1.08s/it]


Timer unit: 1e-09 s

Total time: 70.757 s
File: /var/folders/gq/bts52kyn0v72cpz5c5489984006lvv/T/ipykernel_27340/2367849324.py
Function: train_bpe at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def train_bpe(input_path, vocab_size, special_tokens):
     2         1       1000.0   1000.0      0.0      merges = []
     3         1      40000.0  40000.0      0.0      vocab = [bytes([i]) for i in range(256)]
     4         1       3000.0   3000.0      0.0      vocab += [s.encode("utf-8") for s in special_tokens]
     5                                           
     6         1     249000.0 249000.0      0.0      print("Pretokening .......")
     7         1     2.42e+10 2.42e+10     34.2      token_counter = pre_tokenize_parallel(input_path, special_tokens)
     8                                           
     9         1     194000.0 194000.0      0.0      print("Initializing pairs .......")
    10         1    

In [43]:
vocab_size = 300
merges = []
vocab = [bytes([i]) for i in range(256)]
vocab += [s.encode("utf-8") for s in special_tokens]
init_vocab_len = len(vocab)

token_pair_set = dict()
pair_counter = defaultdict(int)
for token, count in token_counter.items():
    s = set()
    for i in range(len(token) - 1):
        pair = token[i : i + 2]
        pair_counter[pair] += count
        s.add(pair)
    token_pair_set[token] = s

In [46]:
def train():
    print("Training BPE .......")
    for epoch in tqdm(range(init_vocab_len + 1, vocab_size)):
        max_pair = max(pair_counter, key=lambda p: (pair_counter[p], p))
        mp0, mp1 = max_pair
        merged_pair = mp0 + mp1
        merges.append(max_pair)

        for token, count in list(token_counter.items()):
            if max_pair not in token_pair_set[token]:
                continue   # no occurrence — token stays unchanged in dict
            modify = False
            new_token = []
            i = 0
            n = len(token)
            while i < n:
                if i + 1 < n and token[i] == mp0 and token[i + 1] == mp1:
                    modify = True
                    if new_token:                       # left-neighbor update
                        left = new_token[-1]
                        lp = (left, mp0)
                        c = pair_counter[lp] - count
                        if c:
                            pair_counter[lp] = c
                        else:
                            del pair_counter[lp]
                        pair_counter[(left, merged_pair)] += count
                    if i + 2 < n:                       # right-neighbor update
                        right = token[i + 2]
                        rp = (mp1, right)
                        c = pair_counter[rp] - count
                        if c:
                            pair_counter[rp] = c
                        else:
                            del pair_counter[rp]
                        pair_counter[(merged_pair, right)] += count
                    new_token.append(merged_pair)
                    i += 2
                else:
                    new_token.append(token[i])
                    i += 1

            # ── in-place update (no new_counter) ──────────────────────────
            if modify:
                new_key = tuple(new_token)
                del token_counter[token]
                del token_pair_set[token]
                token_counter[new_key] = token_counter.get(new_key, 0) + count
                token_pair_set[new_key] = {
                    (new_token[j], new_token[j + 1]) for j in range(len(new_token) - 1)
                }

        del pair_counter[max_pair]   # max_pair gone from all tokens; remove stale count
        vocab.append(merged_pair)

In [47]:
%lprun -f train train()

Training BPE .......


100%|██████████| 42/42 [00:40<00:00,  1.03it/s]


Timer unit: 1e-09 s

Total time: 37.0051 s
File: /var/folders/gq/bts52kyn0v72cpz5c5489984006lvv/T/ipykernel_81242/3286592027.py
Function: train at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def train():
     2         1      74000.0  74000.0      0.0      print("Training BPE .......")
     3        43   39942000.0 928883.7      0.1      for epoch in tqdm(range(init_vocab_len + 1, vocab_size)):
     4        42  392939000.0 9.36e+06      1.1          max_pair = max(pair_counter, key=lambda p: (pair_counter[p], p))
     5        42      12000.0    285.7      0.0          mp0, mp1 = max_pair
     6        42      39000.0    928.6      0.0          merged_pair = mp0 + mp1
     7        42      35000.0    833.3      0.0          merges.append(max_pair)
     8                                           
     9  26354454     1.63e+10    619.4     44.1          for token, count in list(token_counter.items()):
    10  2

In [ ]:

def train_bpe(input_path, vocab_size, special_tokens):
    merges = []
    vocab = [bytes([i]) for i in range(256)]
    vocab += [s.encode("utf-8") for s in special_tokens]

    print("Pretokening .......")
    token_counter = pre_tokenize_parallel(input_path, special_tokens)

    print("Initializing pairs .......")
    merges = []
    vocab = [bytes([i]) for i in range(256)]
    vocab += [s.encode("utf-8") for s in special_tokens]
    init_vocab_len = len(vocab)

    token_pair_set = dict()
    pair_counter = defaultdict(int)
    for token, count in token_counter.items():
        s = set()
        for i in range(len(token) - 1):
            pair = token[i : i + 2]
            pair_counter[pair] += count
            s.add(pair)
        token_pair_set[token] = s

    print("Training BPE .......")
    for epoch in tqdm(range(init_vocab_len + 1, vocab_size)):
        max_pair = max(pair_counter, key=lambda p: (pair_counter[p], p))
        mp0, mp1 = max_pair
        merged_pair = mp0 + mp1
        merges.append(max_pair)

        for token, count in list(token_counter.items()):
            if max_pair not in token_pair_set[token]:
                continue   # no occurrence — token stays unchanged in dict
            modify = False
            new_token = []
            i = 0
            n = len(token)
            while i < n:
                if i + 1 < n and token[i] == mp0 and token[i + 1] == mp1:
                    modify = True
                    if new_token:                       # left-neighbor update
                        left = new_token[-1]
                        lp = (left, mp0)
                        c = pair_counter[lp] - count
                        if c:
                            pair_counter[lp] = c
                        else:
                            del pair_counter[lp]
                        pair_counter[(left, merged_pair)] += count
                    if i + 2 < n:                       # right-neighbor update
                        right = token[i + 2]
                        rp = (mp1, right)
                        c = pair_counter[rp] - count
                        if c:
                            pair_counter[rp] = c
                        else:
                            del pair_counter[rp]
                        pair_counter[(merged_pair, right)] += count
                    new_token.append(merged_pair)
                    i += 2
                else:
                    new_token.append(token[i])
                    i += 1

            # ── in-place update (no new_counter) ──────────────────────────
            if modify:
                new_key = tuple(new_token)
                del token_counter[token]
                del token_pair_set[token]
                token_counter[new_key] = token_counter.get(new_key, 0) + count
                token_pair_set[new_key] = {
                    (new_token[j], new_token[j + 1]) for j in range(len(new_token) - 1)
                }

        del pair_counter[max_pair]   # max_pair gone from all tokens; remove stale count
        vocab.append(merged_pair)
        
    vocab = {i: x for i, x in enumerate(vocab)}
    return vocab, merges